# Paisagem sonora urbana: sons da cidade e mapa de poluição sonora

**Júlia Roberta Quoos Alves** · ohtopico.com.br

Três perguntas, em ordem:

1. **Paisagem sonora:** que sons aparecem na cidade, em que hora e em que dia? (só com a planilha de anotações, sem baixar áudio)
2. **Sons urbanos:** um modelo consegue reconhecer motor, buzina/sirene, serra, obra, música, voz e cachorro em 10 s de áudio?
3. **Mapa de poluição sonora:** onde a cidade é mais barulhenta e por causa de quê? Primeiro em Nova York (dados prontos), depois com gravações próprias em Santa Maria.

**Dados:** SONYC-UST v2.3 (Cartwright et al., 2020; Zenodo 3966543, CC BY 4.0). São 18.510 gravações de 10 s feitas por 56 sensores fixos em Nova York, com localização (centro do quarteirão), data e hora, e 8 classes grossas / 23 finas.

**Roda no Colab** (GPU não é obrigatória; o YAMNet roda bem em CPU, mas GPU acelera a parte 2).

## 0. Configuração

In [ ]:
!pip -q install folium tensorflow_hub

import os, re, glob, tarfile, urllib.request, json
import numpy as np, pandas as pd
import matplotlib.pyplot as plt

USAR_DRIVE = True   # salva downloads e resultados no Drive para não baixar de novo
if USAR_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE = '/content/drive/MyDrive/paisagem_sonora'
else:
    BASE = '/content/paisagem_sonora'
DADOS = f'{BASE}/sonyc'
SAIDA = f'{BASE}/resultados'
os.makedirs(DADOS, exist_ok=True); os.makedirs(SAIDA, exist_ok=True)

ZENODO = 'https://zenodo.org/records/3966543/files'
def baixar(nome, destino=DADOS):
    caminho = f'{destino}/{nome}'
    if not os.path.exists(caminho):
        print('baixando', nome, '...')
        urllib.request.urlretrieve(f'{ZENODO}/{nome}?download=1', caminho)
    return caminho

# Paleta no estilo do site (ohtopico.com.br)
CORES = ['#7c5cff', '#ff8a3d', '#2bb673', '#ffcc33', '#3fa9f5', '#ff5fa2', '#8d6e63', '#26c6da']
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False})

## 1. Paisagem sonora de Nova York (só anotações, 14,5 MB)

Cada gravação foi anotada por voluntários (Zooniverse, `annotator_id > 0`), pela equipe SONYC (`< 0`) e, em parte, tem uma anotação **verificada** (`annotator_id == 0`). Para cada gravação usamos a verificada quando existe; senão, o voto da maioria dos voluntários.

In [ ]:
anot = pd.read_csv(baixar('annotations.csv'))
print(anot.shape)
print(anot[['split','sensor_id','borough','year']].nunique())

pres_grossa = sorted([c for c in anot.columns if re.fullmatch(r'\d+_[a-z\-]+_presence', c)])
CLASSES = [re.sub(r'^\d+_|_presence$', '', c) for c in pres_grossa]
NOMES_PT = {'engine':'motor', 'machinery-impact':'impacto de máquina (obra)', 'non-machinery-impact':'impacto (batida, porta)',
            'powered-saw':'serra elétrica', 'alert-signal':'buzina/sirene/alarme', 'music':'música',
            'human-voice':'voz humana', 'dog':'cachorro'}
print(CLASSES)

In [ ]:
# Um rótulo por gravação: verificado (annotator_id == 0) se houver, senão maioria dos voluntários
meta_cols = ['split','sensor_id','audio_filename','year','week','day','hour','borough','block','latitude','longitude']
a = anot.copy()
a[pres_grossa] = a[pres_grossa].where(a[pres_grossa] >= 0)   # -1 = não anotado

verif = a[a.annotator_id == 0].groupby('audio_filename')[pres_grossa].max()
volunt = a[a.annotator_id > 0].groupby('audio_filename')[pres_grossa].mean().ge(0.5).astype(float)
rot = volunt.copy()
rot.loc[verif.index] = verif
rot = rot.fillna(0).astype(int)
rot.columns = CLASSES

meta = a.groupby('audio_filename')[meta_cols[:2] + meta_cols[3:]].first()
grav = meta.join(rot, how='inner').reset_index()
grav['verificado'] = grav.audio_filename.isin(verif.index)
grav['fim_de_semana'] = grav.day.isin([5, 6])   # 0 = segunda ... 6 = domingo
print(len(grav), 'gravações;', grav.verificado.sum(), 'com rótulo verificado')
grav[CLASSES].mean().sort_values(ascending=False).rename(index=NOMES_PT).round(3)

In [ ]:
# Quando cada som aparece? (fração das gravações daquela hora que contém o som)
por_hora = grav.groupby('hour')[CLASSES].mean()
fig, ax = plt.subplots(figsize=(11, 4.2))
im = ax.imshow(por_hora.T, aspect='auto', cmap='magma_r')
ax.set_yticks(range(len(CLASSES)), [NOMES_PT[c] for c in CLASSES])
ax.set_xticks(range(0, 24, 2), [f'{h}h' for h in range(0, 24, 2)])
ax.set_title('Paisagem sonora ao longo do dia (Nova York, SONYC-UST)')
plt.colorbar(im, label='fração das gravações'); plt.tight_layout()
plt.savefig(f'{SAIDA}/01_paisagem_por_hora.png', bbox_inches='tight'); plt.show()

comp = grav.groupby('fim_de_semana')[CLASSES].mean().T.rename(index=NOMES_PT, columns={False:'dia útil', True:'fim de semana'})
comp.sort_values('dia útil').plot.barh(figsize=(8, 4.2), color=[CORES[0], CORES[1]])
plt.title('Dia útil × fim de semana'); plt.xlabel('fração das gravações'); plt.tight_layout()
plt.savefig(f'{SAIDA}/02_util_x_fds.png', bbox_inches='tight'); plt.show()

### 1b. Mapa por sensor

Cada ponto é um sensor (a coordenada é o centro do quarteirão). O **índice de incômodo** é a fração das gravações com algum dos sons ligados a reclamações de barulho em NY: motor, obra, serra e buzina/sirene. É uma medida de *presença*, não de decibéis: o SONYC-UST não traz o nível sonoro.

In [ ]:
import folium
INCOMODO = ['engine', 'machinery-impact', 'powered-saw', 'alert-signal']
grav['incomodo'] = grav[INCOMODO].max(axis=1)

sens = grav.groupby('sensor_id').agg(lat=('latitude','first'), lon=('longitude','first'), borough=('borough','first'),
                                     n=('audio_filename','size'), incomodo=('incomodo','mean'))
# (as médias por classe vão num join: 'engine' é nome de argumento do .agg do pandas)
sens = sens.join(grav.groupby('sensor_id')[CLASSES].mean()).reset_index()
sens = sens[sens.n >= 30]   # evita sensores com poucas gravações

def cor(v, vmin, vmax):
    import matplotlib
    return matplotlib.colors.to_hex(plt.cm.magma_r(0.15 + 0.8 * (v - vmin) / max(vmax - vmin, 1e-9)))

m = folium.Map(location=[sens.lat.mean(), sens.lon.mean()], zoom_start=12, tiles='CartoDB positron')
for nome, col in [('índice de incômodo', 'incomodo')] + [(NOMES_PT[c], c) for c in CLASSES]:
    camada = folium.FeatureGroup(name=nome, show=(col == 'incomodo'))
    vmin, vmax = sens[col].min(), sens[col].max()
    for _, s in sens.iterrows():
        dom = max(CLASSES, key=lambda c: s[c])
        folium.CircleMarker([s.lat, s.lon], radius=6 + 14 * (s[col] - vmin) / max(vmax - vmin, 1e-9),
                            color='#111', weight=1.5, fill=True, fill_opacity=0.85, fill_color=cor(s[col], vmin, vmax),
                            tooltip=f"sensor {s.sensor_id} · {nome}: {s[col]:.0%}",
                            popup=f"<b>sensor {s.sensor_id}</b><br>{s.n} gravações<br>incômodo: {s.incomodo:.0%}<br>som mais comum: {NOMES_PT[dom]}").add_to(camada)
    camada.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)
m.save(f'{SAIDA}/mapa_nova_york.html')
m

## 2. Reconhecer sons urbanos (classificador)

Usamos o **YAMNet** (MobileNet V1 treinada no AudioSet, 521 classes) como extrator de características e treinamos só uma cabeça pequena para as 8 classes do SONYC. É a mesma ideia de transferência das MobileNets da dissertação, agora com espectrogramas.

O áudio completo tem 13 GB (19 arquivos). `N_TAR` controla quantos baixar; 3 arquivos (~2,1 GB) já dão alguns milhares de gravações. A avaliação usa a divisão oficial (`train` / `validate`) e a métrica do DCASE (AUPRC micro e macro).

In [ ]:
N_TAR = 3          # aumente (até 19) se quiser mais dados
AUDIO = f'{DADOS}/audio'
os.makedirs(AUDIO, exist_ok=True)
for i in range(N_TAR):
    marca = f'{DADOS}/.extraido_{i}'
    if os.path.exists(marca): continue
    t = baixar(f'audio-{i}.tar.gz')
    with tarfile.open(t) as tf_: tf_.extractall('/content/sonyc_tmp')
    for f in glob.glob('/content/sonyc_tmp/**/*.wav', recursive=True):
        os.replace(f, f'{AUDIO}/{os.path.basename(f)}')
    open(marca, 'w').close()
    if USAR_DRIVE: os.remove(t)   # economiza espaço no Drive
arquivos = {os.path.basename(f) for f in glob.glob(f'{AUDIO}/*.wav')}
base = grav[grav.audio_filename.isin(arquivos)].reset_index(drop=True)
print(len(base), 'gravações com áudio'); print(base.split.value_counts())

In [ ]:
import tensorflow as tf, tensorflow_hub as hub, librosa
yamnet = hub.load('https://tfhub.dev/google/yamnet/1')
NOMES_AUDIOSET = pd.read_csv(yamnet.class_map_path().numpy().decode())['display_name'].tolist()

def carregar(caminho):
    y, _ = librosa.load(caminho, sr=16000, mono=True)
    return y.astype(np.float32)

def embedding(y):
    scores, emb, _ = yamnet(y)
    emb = emb.numpy()
    return np.concatenate([emb.mean(0), emb.max(0)]), scores.numpy().mean(0)

cache = f'{DADOS}/emb_yamnet.npz'
feito = dict(np.load(cache, allow_pickle=True)) if os.path.exists(cache) else {}
from tqdm.auto import tqdm
for f in tqdm(base.audio_filename):
    if f not in feito:
        feito[f] = embedding(carregar(f'{AUDIO}/{f}'))[0]
np.savez(cache, **feito)
X = np.stack([feito[f] for f in base.audio_filename]); Y = base[CLASSES].values
print(X.shape, Y.shape)

In [ ]:
from sklearn.metrics import average_precision_score, f1_score
tr, va = (base.split == 'train').values, (base.split == 'validate').values
mu, sd = X[tr].mean(0), X[tr].std(0) + 1e-6
Xn = (X - mu) / sd

tf.keras.utils.set_random_seed(42)
cabeca = tf.keras.Sequential([tf.keras.Input((X.shape[1],)), tf.keras.layers.Dropout(0.3),
                              tf.keras.layers.Dense(256, activation='relu'), tf.keras.layers.Dropout(0.3),
                              tf.keras.layers.Dense(len(CLASSES), activation='sigmoid')])
cabeca.compile('adam', 'binary_crossentropy', metrics=[tf.keras.metrics.AUC(curve='PR', multi_label=True, name='auprc')])
cabeca.fit(Xn[tr], Y[tr], validation_data=(Xn[va], Y[va]), epochs=60, batch_size=64, verbose=0,
           callbacks=[tf.keras.callbacks.EarlyStopping('val_auprc', mode='max', patience=8, restore_best_weights=True)])

P = cabeca.predict(Xn[va], verbose=0)
validas = [k for k in range(len(CLASSES)) if Y[va][:, k].sum() > 0]
res = pd.DataFrame({'classe': [NOMES_PT[CLASSES[k]] for k in validas],
                    'positivos': Y[va][:, validas].sum(0),
                    'AUPRC': [average_precision_score(Y[va][:, k], P[:, k]) for k in validas],
                    'F1@0,5': [f1_score(Y[va][:, k], P[:, k] > 0.5) for k in validas]}).round(3)
print('AUPRC micro:', round(average_precision_score(Y[va][:, validas], P[:, validas], average='micro'), 3),
      '| macro:', round(res.AUPRC.mean(), 3))
res.to_csv(f'{SAIDA}/metricas_classificador.csv', index=False); res

In [ ]:
cabeca.save(f'{SAIDA}/cabeca_sonyc.keras')
np.savez(f'{SAIDA}/normalizacao.npz', mu=mu, sd=sd, classes=np.array(CLASSES))

## 3. Santa Maria: gravações próprias → mapa de poluição sonora

**Como coletar** (celular serve):

1. Em cada ponto, grave **30 s a 1 min** parado, celular na mão, longe do corpo, sem falar. Repita o mesmo ponto em horários diferentes (ex.: 8h, 12h, 18h, 22h; dia útil e fim de semana).
2. Anote numa planilha `pontos.csv` com as colunas: `arquivo,lat,lon,data_hora,local` (ex.: `p01_0800.m4a,-29.7175,-53.7166,2026-11-10 08:00,Calçadão`). Latitude e longitude saem do Google Maps (toque e segure no ponto).
3. Coloque os áudios e o `pontos.csv` em `MyDrive/paisagem_sonora/santa_maria/`.

**Decibéis:** microfone de celular não é calibrado, então o notebook calcula um nível **relativo** ponderado em A (dB(A) aproximado). Para chegar perto de valores reais, meça uma vez com um app de decibelímetro (ex.: NIOSH SLM) junto com a gravação e coloque a diferença em `CALIBRACAO_DB`. Para comparar pontos entre si, isso já basta, desde que use sempre o mesmo celular.

**Privacidade:** as gravações podem captar conversas. O notebook guarda só os níveis e as classes, e não é preciso publicar o áudio.

In [ ]:
SM = f'{BASE}/santa_maria'
CALIBRACAO_DB = 0.0     # dB a somar para aproximar do decibelímetro (medir uma vez)
os.makedirs(SM, exist_ok=True)

def nivel_dba(y, sr=16000, janela_s=1.0):
    # Nível equivalente ponderado em A, por janela de 1 s, em dBFS + calibração
    S = np.abs(librosa.stft(y, n_fft=2048, hop_length=512)) ** 2
    pesoA = librosa.db_to_power(librosa.A_weighting(librosa.fft_frequencies(sr=sr, n_fft=2048)))
    pot = (S * pesoA[:, None]).sum(0) / (2048 ** 2 / 4)
    q = int(janela_s * sr / 512)
    por_s = np.array([pot[i:i + q].mean() for i in range(0, len(pot) - q + 1, q)]) + 1e-12
    db = 10 * np.log10(por_s) + CALIBRACAO_DB
    return {'Leq': 10 * np.log10(np.mean(10 ** (db / 10))), 'L10': np.percentile(db, 90), 'L90': np.percentile(db, 10), 'max': db.max()}

pontos_csv = f'{SM}/pontos.csv'
if not os.path.exists(pontos_csv):
    print('Ainda não há gravações. Coloque os áudios e o pontos.csv em', SM)
else:
    pts = pd.read_csv(pontos_csv)
    linhas = []
    for _, p in tqdm(pts.iterrows(), total=len(pts)):
        y = carregar(f'{SM}/{p.arquivo}')
        trechos = [y[i:i + 160000] for i in range(0, max(len(y) - 160000, 0) + 1, 160000)]   # janelas de 10 s, como no SONYC
        embs, ys = zip(*[embedding(t) for t in trechos])
        prob = cabeca.predict((np.stack(embs) - mu) / sd, verbose=0).max(0)
        top_audioset = [NOMES_AUDIOSET[k] for k in np.mean(ys, 0).argsort()[::-1][:3]]
        linhas.append({**p.to_dict(), **nivel_dba(y), **dict(zip(CLASSES, prob)), 'audioset_top3': ', '.join(top_audioset)})
    sm = pd.DataFrame(linhas)
    sm['hora'] = pd.to_datetime(sm.data_hora).dt.hour
    sm.to_csv(f'{SAIDA}/santa_maria_medicoes.csv', index=False)
    display(sm[['local', 'data_hora', 'Leq', 'L10', 'L90'] + CLASSES].round(2))

In [ ]:
# Mapa de Santa Maria: tamanho/cor = Leq médio do ponto; popup com horários e sons
if os.path.exists(pontos_csv):
    agg = sm.groupby(['local', 'lat', 'lon']).agg(Leq=('Leq', lambda v: 10 * np.log10(np.mean(10 ** (v / 10)))),
                                                  n=('arquivo', 'size'))
    agg = agg.join(sm.groupby(['local', 'lat', 'lon'])[CLASSES].mean()).reset_index()
    msm = folium.Map(location=[agg.lat.mean(), agg.lon.mean()], zoom_start=15, tiles='CartoDB positron')
    vmin, vmax = agg.Leq.min(), agg.Leq.max()
    for _, s in agg.iterrows():
        dom = sorted(CLASSES, key=lambda c: -s[c])[:2]
        horarios = sm[sm.local == s.local].sort_values('hora')
        tab = '<br>'.join(f"{int(h.hora)}h: {h.Leq:.0f} dB" for _, h in horarios.iterrows())
        folium.CircleMarker([s.lat, s.lon], radius=8 + 16 * (s.Leq - vmin) / max(vmax - vmin, 1e-9), color='#111', weight=1.5,
                            fill=True, fill_opacity=0.85, fill_color=cor(s.Leq, vmin, vmax),
                            tooltip=f"{s.local}: {s.Leq:.0f} dB(A) rel.",
                            popup=f"<b>{s.local}</b><br>{tab}<br>sons: {', '.join(NOMES_PT[c] for c in dom)}").add_to(msm)
    msm.save(f'{SAIDA}/mapa_santa_maria.html'); display(msm)

## Referências

- Cartwright, M. et al. (2020). *SONYC-UST-V2: An Urban Sound Tagging Dataset with Spatiotemporal Context*. DCASE Workshop. Dados: doi:10.5281/zenodo.3966543 (CC BY 4.0).
- Gemmeke, J. et al. (2017). *Audio Set*. ICASSP. Modelo YAMNet: tfhub.dev/google/yamnet/1.
- Salamon, J., Jacoby, C., Bello, J. P. (2014). *A Dataset and Taxonomy for Urban Sound Research* (UrbanSound8K). ACM MM.
- ISO 12913-1:2014. *Acoustics: Soundscape, Part 1: Definition and conceptual framework*.